In [ ]:
import os
os.environ["ROGII_GOLD_PROFILE"] = "conservative"

In [ ]:
import sys, os, glob, subprocess
# koolbox setup: wheel install or sys.path fallback
kb_dir = '/kaggle/input/koolbox-offline'
if not os.path.isdir(kb_dir):
    # alt path under datasets/
    cand = glob.glob('/kaggle/input/**/koolbox*', recursive=True)
    print('koolbox candidates:', cand[:5])
    if cand: kb_dir = cand[0]
print('using koolbox dir:', kb_dir)
if os.path.isdir(kb_dir):
    print('listing:', os.listdir(kb_dir)[:20])
    whls = glob.glob(f'{kb_dir}/**/*.whl', recursive=True)
    if whls:
        for w in whls:
            print('install', w)
            subprocess.run(['pip', 'install', '--no-deps', w], check=False)
    else:
        sys.path.insert(0, kb_dir)
        # also try subdirs
        for sub in os.listdir(kb_dir):
            sub_path = os.path.join(kb_dir, sub)
            if os.path.isdir(sub_path):
                sys.path.insert(0, sub_path)
import koolbox
print('koolbox OK:', koolbox.__file__)

In [ ]:
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from koolbox import Trainer
from pathlib import Path
from numba import njit
import matplotlib.pyplot as plt
import multiprocessing
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import joblib
import time
import glob
import os

warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    
    metric = root_mean_squared_error

In [ ]:
# === SMOKE gating (baked-in; Kaggle injects no env). FLIP via generate_notebook.py. ===
SMOKE = False
# Force GBT re-training: the pretrained artifacts were fit on the OLD feature set, so
# they MUST be retrained once we inject the surface + dip-beam columns (else the loaded
# models ignore / mismatch the new features). The train.csv fast-load is untouched; only
# the lightgbm/catboost load-vs-fit branches honor this flag.
FORCE_RETRAIN = True
# Force fleongg (pipeline B) re-training too: its mounted pretrained boosters were fit on
# ALL train wells, so their train-well predictions are in-sample (leak) and main()'s
# INFERENCE branch skips the _BLEND_OOF_STASH. Diagnostic runs need the TRAIN branch
# (GroupKFold OOF). Flip to False when regenerating a submit version that must reproduce
# the banked pretrained-fleongg submission exactly.
FLE_RETRAIN = False
print(f"SMOKE={SMOKE}  FORCE_RETRAIN={FORCE_RETRAIN}  FLE_RETRAIN={FLE_RETRAIN}")


In [ ]:
SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

SELECTOR_BIN_VARIANTS = {
    0: 'pf_scale_5_hold_0.2',
    1: 'pf_scale_3_hold_0.15',
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}

SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def load_well(wid, split='train'):
    base = CFG.dataset_path / split
    hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
    tw = pd.read_csv(base / f'{wid}__typewell.csv')
    return hw, tw


def run_particle_filter(hw, tw, n_particles=500, seed=42, MOM=0.998, VN=0.002, PN=0.005, RP=0.1, RR=0.001, RESAMP=0.5, POS0=4.5, RATE0=0.01, GS_LO=10.0):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), GS_LO, 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + POS0 * rng.standard_normal(N)  # sp45 patch (sel15 vb best)
    rate = ir + RATE0 * rng.standard_normal(N)
    w    = np.ones(N) / N

    # motion constants are kwargs now (v2 PF sweep); defaults == the original literals

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0, **pf_kw):
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s, **pf_kw)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128, **pf_kw):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s, **pf_kw)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
    out['pf_mean'] = pred_arr.mean(0)
    return out


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    parts = name.split('_')
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    base = pf_by_scale.get(f'pf_scale_{scale:g}')
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred

In [ ]:
SEED=42
NCPU=min(4,multiprocessing.cpu_count())

FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

BEAMS=[
    (10,20.0,144.0,2,"cons"),
    (10, 8.0, 64.0,2,"loose"),
    ( 8,35.0,220.0,1,"vcons"),
    (10,14.0, 90.0,5,"sm5"),
    (20, 4.0, 36.0,3,"vloose"),
    (12,12.0,100.0,3,"mid"),
    (15,25.0,180.0,2,"stiff"),
]

PF_N=600; ANCC_N=600
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search Ã‚Â±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # Ã‚Â±2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

# Dense grid for O(1) typewell lookup
def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw,tw_tvt,tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))

def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)

def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                      ev['GR'].values.astype(np.float64),gg,gmin,gst,
                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   ev['GR'].values.astype(np.float64),
                   gr_sm.loc[ev.index].values.astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,N,
                   PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)


_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)

def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

class FormationPlaneKNN:
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")

_FI=FI; _DI=DI
ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)

def build_well(hw_path,tw_path,is_train):
    global _FI,_DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
    if len(pf_a)==0: return None
    pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
    pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
    has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # 7 beams (Numba JIT Ã‚Â±2)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

    # Multi-scale NCC Ã¢â€ â€™ score-weighted ensemble
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        'pf_ancc':pf_use,'pf_ancc_std':std_use,
        'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
        'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    for k,v in rolls.items(): feats[k]=v
    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result

def build_dataset(paths,is_train,label):
    args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
          for p in paths
          if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
    t0=time.time()
    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
        delayed(build_well)(hp,tp,it) for hp,tp,it in args)
    parts=[r for r in res if r is not None]
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

In [ ]:
if (CFG.artifacts_path / "data" / "train.csv").exists():
    train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
else:
    train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    train_df = build_dataset(train_paths, is_train=True, label="train")    

test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
test_df = build_dataset(test_paths, is_train=False, label="test")

features = [c for c in train_df.columns if c not in {'well','id','target'}]

X = train_df[features]
y = train_df['target']
g = train_df['well']

X_test = test_df[features]

In [ ]:

# === Global formation structural-surface features (per-row, datum-separated, LOO fold-safe) ===
from scipy.spatial import cKDTree as _ckd
import numpy as _np, pandas as _pd, gc as _gc, time as _t

TRAIN_DIR = CFG.dataset_path / "train"   # dualpipe path (medal used a TRAIN_DIR constant)
SKIP = {'well', 'id', 'target'}          # dualpipe feature-exclusion set

_SPW = 80   # per-well subsample for the residual field (mirrors DenseANCC SPW)
_KR  = 16   # residual IDW neighbors


def _phi(x, y):
    # 2nd-order polynomial basis (normalized inputs)
    return _np.column_stack([_np.ones_like(x), x, y, x * x, y * y, x * y])


class _FormSurf:
    """Per-formation S_f(X,Y) = global 2nd-order WLS trend + local residual IDW.
       LOO fold-safe: trend via per-well normal-equation downdate, residual via
       self_wid exclusion in the kNN tree AND residuals taken against the SAME
       downdated trend (exact leave-one-well-out, no beta0 imprint)."""

    def __init__(self, wids, data_dir):
        XX = []; YY = []; WW = []; FF = {f: [] for f in FORMATIONS}
        sX = []; sY = []; sW = []; sFd = {f: [] for f in FORMATIONS}
        code = {}
        for wid in wids:
            p = data_dir / (wid + "__horizontal_well.csv")
            try:
                df = _pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            c = code.setdefault(wid, len(code))
            X = df["X"].to_numpy(_np.float64); Y = df["Y"].to_numpy(_np.float64)
            XX.append(X); YY.append(Y); WW.append(_np.full(len(df), c, dtype=_np.int32))
            for f in FORMATIONS:
                FF[f].append(df[f].to_numpy(_np.float64))
            ix = _np.linspace(0, len(df) - 1, min(_SPW, len(df))).astype(int)
            sX.append(X[ix]); sY.append(Y[ix]); sW.append(_np.full(len(ix), c, dtype=_np.int32))
            for f in FORMATIONS:
                sFd[f].append(df[f].to_numpy(_np.float64)[ix])
        self.code = code
        self.X = _np.concatenate(XX); self.Y = _np.concatenate(YY); self.W = _np.concatenate(WW)
        self.F = {f: _np.concatenate(FF[f]) for f in FORMATIONS}
        self.mx = float(self.X.mean()); self.my = float(self.Y.mean())
        self.nx = float(self.X.std() or 1.0); self.ny = float(self.Y.std() or 1.0)
        P = _phi((self.X - self.mx) / self.nx, (self.Y - self.my) / self.ny)
        self.A = P.T @ P + 1e-6 * _np.eye(6)
        self.b = {f: P.T @ self.F[f] for f in FORMATIONS}
        self.Aw = {}; self.bw = {f: {} for f in FORMATIONS}
        for c in _np.unique(self.W):
            m = self.W == c; Pm = P[m]
            self.Aw[int(c)] = Pm.T @ Pm
            for f in FORMATIONS:
                self.bw[f][int(c)] = Pm.T @ self.F[f][m]
        self.beta0 = {f: _np.linalg.solve(self.A, self.b[f]) for f in FORMATIONS}
        # residual tree (subsample); keep RAW formation values for exact-LOO residual
        self.sX = _np.concatenate(sX); self.sY = _np.concatenate(sY); self.sW = _np.concatenate(sW)
        self.sF = {f: _np.concatenate(sFd[f]) for f in FORMATIONS}
        self.scale = _np.array([self.sX.std() or 1.0, self.sY.std() or 1.0])
        self.tree = _ckd(_np.column_stack([self.sX, self.sY]) / self.scale)

    def _beta(self, f, code):
        if code is not None and code in self.Aw:
            return _np.linalg.solve(self.A - self.Aw[code] + 1e-6 * _np.eye(6),
                                    self.b[f] - self.bw[f][code])
        return self.beta0[f]

    def _trend(self, beta, xn, yn):
        return (beta[0] + beta[1] * xn + beta[2] * yn
                + beta[3] * xn * xn + beta[4] * yn * yn + beta[5] * xn * yn)

    def predict(self, xy, wid=None):
        code = self.code.get(wid) if wid is not None else None
        xy = _np.atleast_2d(xy)
        xq = (xy[:, 0] - self.mx) / self.nx; yq = (xy[:, 1] - self.my) / self.ny
        kf = min(_KR + 8, len(self.sX))
        dist, idx = self.tree.query(xy / self.scale, k=kf, workers=-1)
        dist = _np.atleast_2d(dist); idx = _np.atleast_2d(idx)
        if code is not None:
            dist = _np.where(self.sW[idx] == code, _np.inf, dist)
        kk = min(_KR - 1, kf - 1)
        ordr = _np.argpartition(dist, kk, 1)[:, :_KR]
        dk = _np.take_along_axis(dist, ordr, 1); ik = _np.take_along_axis(idx, ordr, 1)
        vk = _np.isfinite(dk); w = _np.where(vk, 1.0 / (dk + 1e-3), 0.0)
        ws = w.sum(1); safe = _np.where(ws < 1e-9, 1.0, ws)
        nbx = (self.sX[ik] - self.mx) / self.nx; nby = (self.sY[ik] - self.my) / self.ny
        out = {}
        for f in FORMATIONS:
            beta = self._beta(f, code)
            trend_q = self._trend(beta, xq, yq)
            # exact-LOO residual: neighbor residuals against the SAME downdated trend
            resid_nb = self.sF[f][ik] - self._trend(beta, nbx, nby)
            resid = (resid_nb * w).sum(1) / safe
            out[f] = (trend_q + resid).astype(_np.float64)
        return out


print("[SURF] building FormationSurfaceModel ..."); _t0 = _t.time()
_FS = _FormSurf(train_wids, TRAIN_DIR)
print(f"[SURF]  trend pts={len(_FS.X):,} tree pts={len(_FS.sX):,} ({_t.time()-_t0:.0f}s)")


def _surf_feats(paths, is_train):
    recs = []
    for p in paths:
        wid = p.stem.replace("__horizontal_well", "")
        try:
            hw = _pd.read_csv(p)
        except Exception:
            continue
        if "TVT_input" not in hw.columns:
            continue
        kn = hw[hw["TVT_input"].notna()]
        ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0 or len(kn) < 10:
            continue
        last_tvt = float(kn["TVT_input"].to_numpy()[-1])
        swid = wid if is_train else None
        xk = kn[["X", "Y"]].to_numpy(_np.float64); zk = kn["Z"].to_numpy(_np.float64)
        tk = kn["TVT_input"].to_numpy(_np.float64)
        xe = ev[["X", "Y"]].to_numpy(_np.float64); ze = ev["Z"].to_numpy(_np.float64)
        Sk = _FS.predict(xk, swid); Se = _FS.predict(xe, swid)
        tvts = {}; bs = []
        for f in FORMATIONS:
            b_f = float(_np.median(tk + zk - Sk[f]))
            bs.append(b_f)
            tvts[f] = (-ze + Se[f] + b_f)
        # clip per well to its own vertical scale (guards quadratic-trend blow-up
        # outside the train convex hull on unseen wells); all in delta units.
        rng = float(tk.max() - tk.min())
        cap = 3.0 * rng + 1000.0
        deltas = {f: _np.clip(tvts[f] - last_tvt, -cap, cap) for f in FORMATIONS}
        M = _np.stack([deltas[f] for f in FORMATIONS], 1)  # (n,6) clipped deltas
        smean_d = M.mean(1); sstd = M.std(1); srng = M.max(1) - M.min(1)
        bs = _np.array(bs); b_spread = float(bs.std())
        ids = [f"{wid}_{i}" for i in ev.index]
        d = {"id": ids,
             "surf_mean_d": smean_d.astype(_np.float32),
             "surf_std": sstd.astype(_np.float32),
             "surf_rng": srng.astype(_np.float32),
             "surf_datum_spread": _np.full(len(ev), b_spread, _np.float32)}
        for f in FORMATIONS:
            d[f"tvtS_{f}_d"] = deltas[f].astype(_np.float32)
        recs.append(_pd.DataFrame(d))
    return _pd.concat(recs, ignore_index=True) if recs else _pd.DataFrame({"id": []})


print("[SURF] train surface feats (LOO) ..."); _t0 = _t.time()
_sf_tr = _surf_feats(hw_paths, True)
print(f"[SURF]  train surf rows={len(_sf_tr)} ({_t.time()-_t0:.0f}s)")
print("[SURF] test surface feats ..."); _t0 = _t.time()
_sf_te = _surf_feats(test_paths, False)
print(f"[SURF]  test surf rows={len(_sf_te)} ({_t.time()-_t0:.0f}s)")

_n_tr0 = len(train_df); _n_te0 = len(test_df)
train_df = train_df.merge(_sf_tr, on="id", how="left")
test_df = test_df.merge(_sf_te, on="id", how="left")
assert len(train_df) == _n_tr0 and len(test_df) == _n_te0, "surf merge changed row count"
_surfcols = [c for c in _sf_tr.columns if c != "id"]
_miss_tr = int(train_df[_surfcols].isna().any(axis=1).sum())
_miss_te = int(test_df[_surfcols].isna().any(axis=1).sum())
print(f"[SURF] merged surf cols={len(_surfcols)} | train NaN-rows={_miss_tr} test NaN-rows={_miss_te}")
assert _miss_tr == 0, f"[SURF] {_miss_tr} train rows lack surface features (id set mismatch with build_well)"
for c in _surfcols:
    train_df[c] = train_df[c].fillna(0.0).astype(_np.float32)
    test_df[c] = test_df[c].fillna(0.0).astype(_np.float32)

# recompute the model matrices so the GBT + Ridge stack (and the later TCN) see the surface features
features = [c for c in train_df.columns if c not in SKIP]
X = train_df[features]; y = train_df["target"]; g = train_df["well"]
X_test = test_df[features]
print(f"[SURF] #features now {len(features)} (+{len(_surfcols)} surface)")
_gc.collect()


In [ ]:

# === dip-aware beam member (intra-well, fold-safe by construction) ===
import numpy as _qnp, pandas as _qpd, gc as _qgc
from pathlib import Path as _QPath
from numba import njit as _qnjit

SKIP = {'well', 'id', 'target'}

@_qnjit(cache=True)
def _beam_dip_jit(sgr, tw_gr, si, BS, mc, es, d_exp, W):
    n=len(sgr); nt=len(tw_gr); MAX=BS*(2*W+6)
    bidx=_qnp.zeros(BS,_qnp.int64); bidx[0]=si
    bcost=_qnp.full(BS,1e30); bcost[0]=0.; bn=_qnp.int64(1)
    hI=_qnp.zeros((n,BS),_qnp.int64); hP=_qnp.zeros((n,BS),_qnp.int64)
    cI=_qnp.zeros(MAX,_qnp.int64); cC=_qnp.full(MAX,1e30); cP=_qnp.zeros(MAX,_qnp.int64)
    for step in range(n):
        gv=sgr[step]; de=d_exp[step]; nc=_qnp.int64(0)
        dlo=int(_qnp.floor(de))-W; dhi=int(_qnp.ceil(de))+W
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(dlo,dhi+1):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                dd=d-de
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(dd if dd>=0 else -dd)
                fnd=_qnp.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        if nc==0:
            bp=_qnp.int64(0)
            for bi in range(1,bn):
                if bcost[bi]<bcost[bp]: bp=bi
            ci0=bidx[bp]
            if ci0<0: ci0=_qnp.int64(0)
            if ci0>=nt: ci0=_qnp.int64(nt-1)
            cI[0]=ci0; cC[0]=bcost[bp]; cP[0]=bp; nc=_qnp.int64(1)
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]; cC[i],cC[mi]=cC[mi],cC[i]; cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=_qnp.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=_qnp.zeros(n,_qnp.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

def _dipbeam_one(hw_path):
    wid=_QPath(hw_path).stem.replace('__horizontal_well','')
    twp=_QPath(hw_path).parent/(wid+'__typewell.csv')
    if not twp.exists(): return None
    try:
        hw=_qpd.read_csv(hw_path); tw=_qpd.read_csv(twp).sort_values('TVT')
    except Exception: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10 or len(tw)<5: return None
    tw_tvt=tw['TVT'].to_numpy(_qnp.float64); tw_gr=tw['GR'].to_numpy(_qnp.float64)
    dtvt=float(_qnp.median(_qnp.diff(tw_tvt)))
    if dtvt<=0: return None
    ktvt=kn['TVT_input'].to_numpy(_qnp.float64); kz=kn['Z'].to_numpy(_qnp.float64)
    kx=kn['X'].to_numpy(_qnp.float64); ky=kn['Y'].to_numpy(_qnp.float64)
    # heel dip = d(TVT+Z)/d(horizontal) over the KNOWN zone (known at test time => leak-free)
    dhk=_qnp.hypot(_qnp.diff(kx),_qnp.diff(ky)); hck=_qnp.concatenate([[0.0],_qnp.cumsum(dhk)])
    dip=0.0 if _qnp.std(hck)<1e-6 else float(_qnp.polyfit(hck,ktvt+kz,1)[0])
    last_tvt=float(ktvt[-1])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(_qnp.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(_qnp.float64)        # toe is trailing-contiguous (base pattern)
    sgr=_smooth(hgr,float(_qnp.nanmean(tw_gr)),2).astype(_qnp.float64)
    zx=ev['X'].to_numpy(_qnp.float64); zy=ev['Y'].to_numpy(_qnp.float64); ze=ev['Z'].to_numpy(_qnp.float64)
    Xc=_qnp.concatenate([[float(kx[-1])],zx]); Yc=_qnp.concatenate([[float(ky[-1])],zy]); Zc=_qnp.concatenate([[float(kz[-1])],ze])
    dZt=_qnp.diff(Zc); dht=_qnp.hypot(_qnp.diff(Xc),_qnp.diff(Yc))
    dexp_idx=((-dZt+dip*dht)/dtvt).astype(_qnp.float64)          # len = len(ev)
    si=_nn(tw_tvt,last_tvt)
    path=_beam_dip_jit(sgr,tw_gr,si,10,20.0,144.0,dexp_idx,2)
    tvt_dip=tw_tvt[path]
    ids=[f'{wid}_{i}' for i in ev.index]
    return _qpd.DataFrame({'id':ids,
        'tvt_dipbeam_d':(tvt_dip-last_tvt).astype(_qnp.float32),
        'dipbeam_dip':_qnp.full(len(ev),_qnp.float32(dip),_qnp.float32)})

def _dipbeam_feats(paths):
    recs=[r for r in (_dipbeam_one(p) for p in paths) if r is not None]
    return _qpd.concat(recs,ignore_index=True) if recs else _qpd.DataFrame({'id':[]})

print("[DIPBEAM] computing dip-aware beam member (intra-well) ...")
_db_tr=_dipbeam_feats(hw_paths); _db_te=_dipbeam_feats(test_paths)
_dn0=len(train_df); _dm0=len(test_df)
train_df=train_df.merge(_db_tr,on='id',how='left'); test_df=test_df.merge(_db_te,on='id',how='left')
assert len(train_df)==_dn0 and len(test_df)==_dm0, "[DIPBEAM] merge changed row count"
for _c in ['tvt_dipbeam_d','dipbeam_dip']:
    train_df[_c]=train_df[_c].fillna(0.0).astype(_qnp.float32)
    test_df[_c]=test_df[_c].fillna(0.0).astype(_qnp.float32)
features=[c for c in train_df.columns if c not in SKIP]
X=train_df[features]; y=train_df["target"]; g=train_df["well"]; X_test=test_df[features]
print(f"[DIPBEAM] +2 features | #features now {len(features)}")
_qgc.collect()


In [ ]:
lgb_params = [
    dict(
        boosting_type="gbdt", 
        num_leaves=255, 
        min_child_samples=15,
        subsample=0.8, 
        subsample_freq=1, 
        colsample_bytree=0.8,
        reg_lambda=3.0, 
        reg_alpha=0.05, 
        objective="regression",
        verbose=-1, 
        n_jobs=-1, 
        device_type="gpu", 
        gpu_use_dp=False, 
        max_bin=255,
        learning_rate=0.030, 
        n_estimators=5000, 
        seed=123
    ),
    dict(
        n_jobs=-1, 
        verbose=-1, 
        reg_alpha=10.788188919840913, 
        subsample=0.47437582748953966, 
        num_leaves=64, 
        reg_lambda=95.75401894533888, 
        n_estimators=10000,
        random_state=0,
        boosting_type='gbdt', 
        learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, 
        min_child_samples=40,
        device='gpu',
    ),
    dict(
        n_jobs=-1, 
        verbose=-1, 
        reg_alpha=10.788188919840913, 
        subsample=0.47437582748953966, 
        num_leaves=64, 
        reg_lambda=95.75401894533888, 
        n_estimators=10000,
        random_state=29,
        boosting_type='gbdt', 
        learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, 
        min_child_samples=40,
        device='gpu',
    ),
]

cb_params = [
    dict(
        iterations=8000, 
        depth=7, 
        l2_leaf_reg=2.0,
        min_data_in_leaf=15, 
        border_count=254,
        loss_function="RMSE", 
        task_type="GPU", 
        devices="0",
        od_type="Iter", 
        od_wait=300, 
        verbose=0,
        learning_rate=0.020, 
        random_seed=7
    ),
    dict(
        iterations=8000, 
        depth=7, 
        l2_leaf_reg=2.0,
        min_data_in_leaf=15, 
        border_count=254,
        loss_function="RMSE", 
        task_type="GPU", 
        devices="0",
        od_type="Iter", 
        od_wait=300, 
        verbose=0,
        learning_rate=0.030, 
        random_seed=123
    ),
]

ridge_params = {
    "random_state": 42,
    "alpha": 1.6602834637650032,
    "tol": 0.0005030247295617308,
    "positive": True,
    "fit_intercept": True
}

pp_params = {
    'alpha': 1.0,
    'tau': 85,
    'w_pf': 0.09
}

In [ ]:

# === sp45 GBT device fallback (GPU quota / CPU kernel) + SMOKE tree shrink ===
import subprocess as _sp_gpu
_HAS_GPU = False
try:
    _HAS_GPU = _sp_gpu.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    _HAS_GPU = False
if not _HAS_GPU:
    for _p in lgb_params:
        _p.pop("device_type", None); _p.pop("device", None); _p.pop("gpu_use_dp", None)
    for _p in cb_params:
        _p["task_type"] = "CPU"; _p.pop("devices", None)
    print("[GBT-DEVICE] no NVIDIA GPU -> stripped GPU keys (CPU fit)")
else:
    print("[GBT-DEVICE] NVIDIA GPU present -> keeping GPU params")
if SMOKE:
    for _p in lgb_params:
        _p["n_estimators"] = min(_p.get("n_estimators", 5000), 300)
    for _p in cb_params:
        _p["iterations"] = min(_p.get("iterations", 8000), 300)
    print("[GBT-DEVICE] SMOKE -> capped lgb n_estimators / cb iterations at 300")


In [ ]:
oof_preds = {}
test_preds = {}

overall_scores = {}
fold_scores = {}

In [ ]:
for i, params in enumerate(lgb_params):   
    save_path = f"models/lightgbm-{i+1}"
    
    if (CFG.artifacts_path / save_path).exists() and not FORCE_RETRAIN:
        print(f"Loading lightgbm-{i+1} from disk...")
        
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        
        print(f"Loaded lightgbm-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
    else:
     
        trainer = Trainer(
            estimator=LGBMRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path
        )
        
        trainer.fit(
            X, 
            y,
            fit_args={
                "eval_metric": "rmse",
                "callbacks": [
                    log_evaluation(period=250), 
                    early_stopping(stopping_rounds=250)
                ]
            }
        )
        print("\n\n")

    oof_preds[f"lightgbm-{i+1}"] = trainer.oof_preds
    test_preds[f"lightgbm-{i+1}"] = trainer.predict(X_test)
    overall_scores[f"lightgbm-{i+1}"] = trainer.overall_score
    fold_scores[f"lightgbm-{i+1}"] = trainer.fold_scores

In [ ]:
for i, params in enumerate(cb_params):    
    save_path = f"models/catboost-{i+1}"
    if (CFG.artifacts_path / save_path).exists() and not FORCE_RETRAIN:
        print(f"Loading catboost-{i+1} from disk...")
        
        trainer_paths = (CFG.artifacts_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        
        print(f"Loaded catboost-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
    else:
        trainer = Trainer(
            estimator=CatBoostRegressor(**params),
            task="regression",
            metric=CFG.metric,
            cv=CFG.cv,
            cv_args={"groups": g},
            use_early_stopping=True,
            verbose=True,
            save=True,
            save_path=save_path
        )
        
        trainer.fit(
            X, 
            y,
            fit_args={
                "verbose": 250,
                "early_stopping_rounds": 250,
                "use_best_model": True
            }
        )
        print("\n\n")

    oof_preds[f"catboost-{i+1}"] = trainer.oof_preds
    test_preds[f"catboost-{i+1}"] = trainer.predict(X_test)
    overall_scores[f"catboost-{i+1}"] = trainer.overall_score
    fold_scores[f"catboost-{i+1}"] = trainer.fold_scores

In [ ]:

# === TCN sequence model as a Ridge-stack member (3rd, decorrelated; per-row features) ===
import torch, torch.nn as nn
import gc
torch.manual_seed(42); np.random.seed(42)
_TE = 2 if SMOKE else 40
_TPAT = 1 if SMOKE else 8
_TCH = 32 if SMOKE else 128
_TNB = 2 if SMOKE else 7
_TDROP, _TLR, _TWD, _TCLIP = 0.15, 5e-4, 1e-4, 1.0
_NSEED = 1 if SMOKE else 3
_tfeat = list(features)
print(f"[TCN] SMOKE={SMOKE} ep={_TE} ch={_TCH} nb={_TNB} nfeat={len(_tfeat)} seeds={_NSEED}")

_dev = "cpu"
try:
    if torch.cuda.is_available():
        _tg = torch.zeros(8, device="cuda"); _ = float((_tg + 1).sum().item()); _dev = "cuda"
        print("[TCN] GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("[TCN] GPU unusable -> CPU:", str(e)[:100])

_Xall = train_df[_tfeat].to_numpy(np.float32); _Xall[~np.isfinite(_Xall)] = np.nan
_mu = np.nanmean(_Xall, 0).astype(np.float32); _sd = np.nanstd(_Xall, 0).astype(np.float32); _sd[_sd < 1e-6] = 1.0
del _Xall; gc.collect()
_yt = train_df['target'].to_numpy(np.float32); _ymu = float(np.nanmean(_yt)); _ysd = float(np.nanstd(_yt)) or 1.0


def _tnorm(M):
    M = M.astype(np.float32, copy=True); M[~np.isfinite(M)] = np.nan; M = (M - _mu) / _sd
    return np.nan_to_num(M, nan=0.0, posinf=0.0, neginf=0.0)


def _seqs(df, has_t):
    df = df.copy(); df['_ri'] = df['id'].str.rsplit('_', n=1).str[-1].astype(int)
    out = []
    for wid, gd in df.groupby('well', sort=False):
        gd = gd.sort_values('_ri')
        it = {'wid': wid, 'X': _tnorm(gd[_tfeat].to_numpy()), 'ids': gd['id'].to_numpy()}
        if has_t:
            it['t'] = (gd['target'].to_numpy(np.float32) - _ymu) / _ysd
        out.append(it)
    return out


_tr = _seqs(train_df, True); _te = _seqs(test_df, False)
_grp = np.array([s['wid'] for s in _tr])


def _gn(c):
    return nn.GroupNorm(8 if c % 8 == 0 else 1, c)


class _Blk(nn.Module):
    def __init__(s, c, d, dr):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, 3, padding=d, dilation=d); s.n1 = _gn(c)
        s.c2 = nn.Conv1d(c, c, 3, padding=d, dilation=d); s.n2 = _gn(c)
        s.a = nn.ReLU(); s.do = nn.Dropout(dr)

    def forward(s, x):
        y = s.do(s.a(s.n1(s.c1(x)))); y = s.n2(s.c2(y)); return s.a(x + y)


class _TCN(nn.Module):
    def __init__(s, ci, c, nb, dr):
        super().__init__()
        s.inp = nn.Conv1d(ci, c, 1)
        s.bl = nn.ModuleList([_Blk(c, 2 ** i, dr) for i in range(nb)])
        s.h = nn.Conv1d(c, 1, 1)

    def forward(s, x):
        x = s.inp(x)
        for b in s.bl:
            x = b(x)
        return s.h(x).squeeze(1)


def _hub(p, t, d=1.0):
    e = p - t; a = e.abs(); return torch.where(a <= d, 0.5 * e * e, d * (a - 0.5 * d)).mean()


def _tx(s):
    return torch.tensor(s['X'].T[None], dtype=torch.float32, device=_dev)


def _train_one(_tri, _vai, _seed):
    torch.manual_seed(_seed); np.random.seed(_seed)
    _m = _TCN(len(_tfeat), _TCH, _TNB, _TDROP).to(_dev)
    _opt = torch.optim.Adam(_m.parameters(), lr=_TLR, weight_decay=_TWD)
    _sch = torch.optim.lr_scheduler.CosineAnnealingLR(_opt, T_max=_TE)
    _best = 1e9; _bs = None; _bad = 0; _trl = np.array(_tri)
    for _ep in range(_TE):
        _m.train(); np.random.shuffle(_trl)
        for _j in _trl:
            s = _tr[_j]; x = _tx(s); t = torch.tensor(s['t'][None], dtype=torch.float32, device=_dev)
            _opt.zero_grad(); _l = _hub(_m(x), t); _l.backward()
            torch.nn.utils.clip_grad_norm_(_m.parameters(), _TCLIP); _opt.step()
        _sch.step()
        _m.eval(); _P = []; _T = []
        with torch.no_grad():
            for _j in _vai:
                s = _tr[_j]; _P.append(_m(_tx(s)).cpu().numpy()[0] * _ysd + _ymu); _T.append(s['t'] * _ysd + _ymu)
        _vr = float(np.sqrt(np.mean((np.concatenate(_P) - np.concatenate(_T)) ** 2)))
        if _vr < _best - 1e-4:
            _best = _vr; _bad = 0
            _bs = {k: v.detach().cpu().clone() for k, v in _m.state_dict().items()}
        else:
            _bad += 1
        if _bad >= _TPAT:
            break
    _m.load_state_dict(_bs); _m.eval(); return _m


_oof_id = {}; _test_sum = {}; _nf = 0; _fb = []
_idx = np.arange(len(_tr))
for _f, (_tri, _vai) in enumerate(GroupKFold(n_splits=CFG.n_splits).split(_idx, groups=_grp)):
    _va_sum = {}; _te_fold = {}
    for _seed in range(_NSEED):
        _m = _train_one(_tri, _vai, 1000 * _seed + _f)
        with torch.no_grad():
            for _j in _vai:
                s = _tr[_j]; pr = _m(_tx(s)).cpu().numpy()[0] * _ysd + _ymu
                for _i, _id in enumerate(s['ids']):
                    _va_sum[_id] = _va_sum.get(_id, 0.0) + float(pr[_i]) / _NSEED
            for s in _te:
                pr = _m(_tx(s)).cpu().numpy()[0] * _ysd + _ymu
                for _i, _id in enumerate(s['ids']):
                    _te_fold[_id] = _te_fold.get(_id, 0.0) + float(pr[_i]) / _NSEED
    for _id, v in _va_sum.items():
        _oof_id[_id] = v
    for _id, v in _te_fold.items():
        _test_sum[_id] = _test_sum.get(_id, 0.0) + v
    _nf += 1
    _vp = []; _vt = []
    for _j in _vai:
        s = _tr[_j]; _vp.extend([_va_sum[i] for i in s['ids']]); _vt.extend(list(s['t'] * _ysd + _ymu))
    _fr = float(np.sqrt(np.mean((np.array(_vp) - np.array(_vt)) ** 2))); _fb.append(_fr)
    print(f"[TCN] fold{_f} toe RMSE={_fr:.4f}")
print("[TCN] CV toe RMSE mean =", float(np.mean(_fb)), "| folds", [round(b, 3) for b in _fb])

_tcn_oof = train_df['id'].map(_oof_id).to_numpy(np.float32)
_te_mean = {k: v / _nf for k, v in _test_sum.items()}
_tcn_test = test_df['id'].map(_te_mean).to_numpy(np.float32)
assert len(_tcn_oof) == len(train_df), "TCN OOF length != train_df"
assert len(_tcn_test) == len(test_df), "TCN test length != test_df"
assert not np.isnan(_tcn_oof).any(), "TCN OOF unmapped"
assert not np.isnan(_tcn_test).any(), "TCN test unmapped"

# --- standalone ABSOLUTE-TVT submission (consumed by the blend-level 3-way member) ---
# test_df['last_known_tvt'] is the RAW last-known TVT: build_well stores it via the
# constant broadcast `sc(last_tvt)` (np.full, NOT a scaler), so absolute = last_known + delta.
_lk = test_df['last_known_tvt'].to_numpy(np.float32)
assert len(_lk) == len(_tcn_test), "last_known_tvt / tcn_test length mismatch"
_tcn_abs = (_lk + _tcn_test).astype(float)
import os as _os
_wk = '/kaggle/working' if _os.path.exists('/kaggle/working') else '.'
pd.DataFrame({'id': test_df['id'].astype(str), 'tvt': _tcn_abs}).to_csv(f'{_wk}/tcn_submission.csv', index=False)
print(f"[TCN] wrote tcn_submission.csv abs (rows={len(_tcn_abs)})")

# oof_preds / test_preds are still dicts here -> add the member; the Ridge stack weights it.
# (kept independently of the standalone blend member above; A-ridge gain is free.)
oof_preds["tcn"] = _tcn_oof
test_preds["tcn"] = _tcn_test
print(f"[TCN] member added | OOF residual RMSE={float(np.sqrt(np.mean((_tcn_oof - y.values) ** 2))):.4f} | members={list(oof_preds.keys())}")
del _tr, _te; gc.collect()


In [ ]:
oof_preds = pd.DataFrame(oof_preds)
test_preds = pd.DataFrame(test_preds)

In [ ]:
ridge_trainer = Trainer(
    Ridge(**ridge_params),
    task="regression",
    metric=CFG.metric,
    cv=CFG.cv,
    cv_args={"groups": g},
    verbose=True
)

ridge_trainer.fit(oof_preds, y)

ridge_oof_preds = ridge_trainer.oof_preds
ridge_test_preds = ridge_trainer.predict(test_preds)

overall_scores["ridge"] = ridge_trainer.overall_score
fold_scores["ridge"] = ridge_trainer.fold_scores

In [ ]:

# === DIAG: does the TCN member lower pipeline A's Ridge OOF? (private proxy) ===
from sklearn.linear_model import Ridge as _RD
from sklearn.model_selection import GroupKFold as _GKF


def _ridge_oof_rmse(_cols):
    _M = oof_preds[_cols].to_numpy(); _yv = y.values; _oof = np.zeros(len(_M))
    for _tri, _vai in _GKF(n_splits=CFG.n_splits).split(_M, groups=g):
        _r = _RD(**ridge_params).fit(_M[_tri], _yv[_tri])
        _oof[_vai] = _r.predict(_M[_vai])
    return float(np.sqrt(np.mean((_oof - _yv) ** 2)))


_cols_all = list(oof_preds.columns)
_cols_notcn = [c for c in _cols_all if c != "tcn"]
_r_notcn = _ridge_oof_rmse(_cols_notcn)
_r_tcn = _ridge_oof_rmse(_cols_all)
print(f"[DIAG] Ridge-A OOF  without_TCN={_r_notcn:.4f}  with_TCN={_r_tcn:.4f}  "
      f"delta={_r_tcn - _r_notcn:+.4f}  (negative = TCN helps the base)")
# surface + dip-beam feature count (compare without_TCN to the prior no-surf baseline)
_sdcols = [c for c in features if c.startswith("tvtS_") or c.startswith("surf_")
           or c in ("tvt_dipbeam_d", "dipbeam_dip")]
print(f"[SURF-DIPBEAM] features added: {len(_sdcols)} ({sorted(_sdcols)}) | total features={len(features)}")


In [ ]:
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md * (1-w_pf) + pd_ * w_pf
    if tau: 
        d *= (1.-np.exp(-np.maximum(df['md_since'].values,0.) / tau))
        
    return d * alpha

def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    
    for _, g in df.groupby('well', sort=False):
        v = g[col].values
        n = len(v)
        wl = min(sg_w, n)
        
        if wl % 2 == 0: 
            wl -= 1
            
        if wl >= sg_p + 2: 
            v = savgol_filter(v, wl, sg_p)
            
        df.loc[g.index,col] = v
        
    return df

In [ ]:
base = train_df['last_known_tvt'].values
ytrue = y.values + base

pf_oof = (train_df['pf_ancc'].values - base)

d = apply_pp(train_df, ridge_oof_preds, pf_oof, **pp_params)
ridge_score = root_mean_squared_error(ytrue, base + d)

overall_scores["ridge (pp)"] = ridge_score
fold_scores["ridge (pp)"] = [ridge_score] * CFG.n_splits

In [ ]:
test_df2 = test_df.copy()
pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
    test_df2, 
    ridge_test_preds,
    pf_test, 
    **pp_params
)
test_df2 = sg_smooth(test_df2, 'pred')

In [ ]:
sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
sub_1 = (sample_sub[['id']].merge(
    test_df2[['id', 'pred']].rename(columns={'pred':'tvt'}),
    on='id', 
    how='left'
))

sub_1['tvt']=sub_1['tvt'].fillna(float(train_df['last_known_tvt'].mean()+train_df['target'].mean()))
sub_1

In [ ]:
sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
train_wells = [os.path.basename(f).split('__')[0] for f in train_hw_files]

test_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
test_wells = [os.path.basename(f).split('__')[0] for f in test_hw_files]

rows = []
for i, wid in enumerate(test_wells):
    print(f'\nProcessing {i + 1}/{len(test_wells)}: {wid}...')
    hw_te, tw_te = load_well(wid, 'test')

    tvt_phys = None
    hw_tr    = None
    tw_tr    = None

    # Physical model for visible wells
    if wid in train_wells:
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print(f'  Physical model OK')
        except Exception as e:
            print(f'  Physical model failed: {e}')
            tvt_phys = None

    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

    # 128-seed likelihood-weighted PF ensemble
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=500, n_seeds=128)
        tvt_pf = pf_by_scale['pf_scale_8']
        print(f'  PF 128-seed lik-ensemble OK scales={SELECTOR_SCALES}')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f'pf_scale_{scale:g}': tvt_pf.copy() for scale in SELECTOR_SCALES}

    # Beam search ensemble
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()

    # Selector blending
    last_known = hw_te['TVT_input'].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
    tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
    print(
        f'  Selector code={selector_code} variant={selector_variant} '
        f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
    )

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        if tvt_phys is not None:
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            tvt_val = float(tvt_selector[ridx])
        rows.append({'id': row['id'], 'tvt': tvt_val})
    print(f'  Added {len(ws)} rows')

In [ ]:
sub_2 = pd.DataFrame(rows)

In [ ]:
sub = (
    sub_1.merge(sub_2, on='id', suffixes=('_1', '_2'))
       .assign(tvt=lambda x: 0.3 * x['tvt_1'] + 0.7 * x['tvt_2'])
       [['id', 'tvt']]
)
sub.to_csv("submission.csv", index=False)
sub

In [ ]:
# === robust low-order PROJECTION post-processing (patched degree=4, blend=0.75) (CV-validated: raw PF -0.54, deployed components -0.33) ===
# Runs AFTER the 0.3*ridge+0.7*selector blend writes submission.csv; OVERWRITES it with the projected
# version. Per-well robust deg-5 fit of dU = tvt + Z - anchor vs normalized MD -> denoise jitter +
# down-weight wrong-branch outliers. Deterministic; defensive per-well fallback to raw.
import numpy as _np, pandas as _pd
def _robfit(s, y, deg=5):
    if len(s) < deg + 2:
        return y.copy()
    c = _np.polyfit(s, y, deg)
    for _ in range(4):
        r = y - _np.polyval(c, s)
        sc = _np.median(_np.abs(r)) * 1.4826 + 1e-6
        c = _np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    return _np.polyval(c, s)
try:
    _base = _pd.read_csv("submission.csv")   # the just-written blended submission
    assert set(['id','tvt']).issubset(_base.columns)
    _base['well'] = _base['id'].str[:8]
    _base['row_idx'] = _base['id'].str[9:].astype(int)
    _out = dict(zip(_base['id'].values, _base['tvt'].astype(float).values))
    _n_ok = 0
    for _wid, _g in _base.groupby('well'):
        try:
            _hw = _pd.read_csv(CFG.dataset_path / 'test' / (_wid + '__horizontal_well.csv'))
            _kn = _hw[_hw['TVT_input'].notna()]
            if len(_kn) < 5:
                continue
            _last = _kn.iloc[-1]
            _anchor = float(_last['TVT_input']) + float(_last['Z'])
            _ps = float(_last['MD']); _end = float(_hw['MD'].iloc[-1])
            _gi = _g.sort_values('row_idx')
            _ri = _gi['row_idx'].values
            _Z = _hw['Z'].values[_ri].astype(float)
            _md = _hw['MD'].values[_ri].astype(float)
            _s = (_md - _ps) / max(_end - _ps, 1e-6)
            _tvt = _gi['tvt'].values.astype(float)
            _fit = _robfit(_s, (_tvt + _Z) - _anchor, 4)
            _tvt_fit_full = (_anchor + _fit) - _Z
            _tvt_fit = 0.25 * _tvt + 0.75 * _tvt_fit_full
            if not _np.all(_np.isfinite(_tvt_fit)):
                continue
            for _rid, _val in zip(_gi['id'].values, _tvt_fit):
                _out[_rid] = float(_val)
            _n_ok += 1
        except Exception as _e:
            print('proj fallback', _wid, _e)
    print('projection applied to', _n_ok, 'wells')
    _final = _base[['id']].copy()
    _final['tvt'] = _final['id'].map(_out).astype(float)
    _final[['id','tvt']].to_csv("submission.csv", index=False)
    print('wrote projected submission.csv', _final.shape)
except Exception as _e:
    print('PROJECTION SKIPPED (kept blended submission):', _e)

In [ ]:
fold_scores_df = pd.DataFrame(fold_scores)
overall_scores_df = pd.DataFrame({k: [v] for k, v in overall_scores.items()}).transpose().sort_values(by=0, ascending=True)
order = overall_scores_df.index.tolist()

min_score = overall_scores_df.values.flatten().min()
max_score = overall_scores_df.values.flatten().max()
padding = (max_score - min_score) * 0.5
lower_limit = min_score - padding
upper_limit = max_score + padding

fig, axs = plt.subplots(1, 2, figsize=(15, fold_scores_df.shape[1] * 0.5))

boxplot = sns.boxplot(data=fold_scores_df, order=order, ax=axs[0], orient="h", color="grey")
axs[0].set_title(f"Fold RMSE")
axs[0].set_xlabel("")
axs[0].set_ylabel("")

barplot = sns.barplot(x=overall_scores_df.values.flatten(), y=overall_scores_df.index, ax=axs[1], color="grey")
axs[1].set_title(f"Overall RMSE")
axs[1].set_xlabel("")
axs[1].set_xlim(left=lower_limit, right=upper_limit)
axs[1].set_ylabel("")

for i, (score, model) in enumerate(zip(overall_scores_df.values.flatten(), overall_scores_df.index)):
    color = "cyan" if "ridge" in model.lower() else "grey"
    barplot.patches[i].set_facecolor(color)
    boxplot.patches[i].set_facecolor(color)
    barplot.text(score, i, round(score, 3), va="center")

plt.tight_layout()
plt.show()

In [ ]:
# === PP stub (kernel B runs no fleongg; [BLEND-OOF-FLE] reads PP.w_sub1/sg_win/sg_poly) ===
# Literals copied verbatim from fleongg's `class PP` in the base notebook (code cell 32).
class PP:
    alpha = 1.0
    tau = 85.0
    w_pf = 0.0
    w_sub1 = 0.60
    sub2_scale = "scale_5"
    sg_win = 61
    sg_poly = 3
print("[PP-STUB] w_sub1=", PP.w_sub1, "sg_win=", PP.sg_win, "sg_poly=", PP.sg_poly, flush=True)


In [ ]:

import numpy as _bnp
import pandas as _bpd
try:
    _BO_N     = 8   if SMOKE else 40
    _BO_SEEDS = 24  if SMOKE else 128   # production sp45 test loop: n_seeds=128
    _BO_PART  = 350 if SMOKE else 500   # production sp45 test loop: n_particles=500
    _BO_INNER_GRID = (0.50, 0.60, 0.70, 0.80, 0.90)
    _BO_SP45_GRID  = (0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)
    _BO_TCN_GRID   = (0.00, 0.05, 0.10, 0.15, 0.20, 0.25)

    _BO_DIR = globals().get('_GEO_DIR')
    if _BO_DIR is None:
        _BO_DIR = (CFG.DATA / 'train') if hasattr(CFG, 'DATA') else (CFG.dataset_path / 'train')
    _bo_wids = sorted(p.stem.replace('__horizontal_well', '')
                      for p in _BO_DIR.glob('*__horizontal_well.csv'))
    if len(_bo_wids) > _BO_N:
        if SMOKE:
            _bo_wids = _bo_wids[:_BO_N]              # align with fleongg's SMOKE cap (first 60)
        else:                                        # deterministic even stride (same as GEO-OOF)
            _bstep = len(_bo_wids) / float(_BO_N)
            _bo_wids = [_bo_wids[int(_k * _bstep)] for _k in range(_BO_N)]

    # --- pipeline-A sub_1: ridge-pp OOF TVT with the exact production post-processing ---
    # (lookup dicts restricted to the selected wells: keeps the final-stage RAM footprint
    #  small on a kernel with an OOM history)
    _bo_wset = set(_bo_wids)
    _bo_pfd = train_df['pf_ancc'].values.astype(float) - train_df['last_known_tvt'].values.astype(float)
    _bo_d1 = apply_pp(train_df, ridge_oof_preds, _bo_pfd, **pp_params)
    _bo_m = train_df['well'].astype(str).isin(_bo_wset).to_numpy()
    _bo_a = train_df.loc[_bo_m, ['id', 'well']].copy()
    _bo_a['pred'] = (train_df['last_known_tvt'].values.astype(float) + _bo_d1)[_bo_m]
    _bo_a = sg_smooth(_bo_a, 'pred')
    _bo_sub1 = dict(zip(_bo_a['id'].astype(str), _bo_a['pred'].astype(float)))
    _bo_tcnm = dict(zip(train_df['id'].astype(str).to_numpy()[_bo_m],
                        (train_df['last_known_tvt'].values.astype(float)
                         + oof_preds['tcn'].values.astype(float))[_bo_m]))
    _ST = globals().get('_BLEND_OOF_STASH')
    if _ST is None:                       # kernel B: load kernel A's mounted stash
        import numpy as _np_ld, glob as _glob_ld
        _cand = _glob_ld.glob('/kaggle/input/**/fle_stash.npz', recursive=True)
        if _cand:
            _z = _np_ld.load(_cand[0], allow_pickle=False)
            _ST = {_k: _z[_k] for _k in _z.files}
            print('[BLEND-OOF] mounted stash', _cand[0], 'rows=', len(_ST['id']), flush=True)
        else:
            print('[BLEND-OOF] NO mounted fle_stash.npz -> FINAL/sweep skipped', flush=True)
    _bo_fle = ({_k: _v for _k, _v in zip(_ST['id'], _ST['fle']) if _k[:8] in _bo_wset}
               if _ST is not None else {})
    print(f'[BLEND-OOF] wells={len(_bo_wids)} pf={_BO_SEEDS}x{_BO_PART} '
          f'fle_stash={"yes" if _bo_fle else "NO"} tcn_oof=yes', flush=True)

    _bo_wells = []
    import time as _bo_time
    _bo_t0 = _bo_time.time()
    for _bi, _wid in enumerate(_bo_wids):
        try:
            _hw = _bpd.read_csv(_BO_DIR / (_wid + '__horizontal_well.csv'))
            _tw = _bpd.read_csv(_BO_DIR / (_wid + '__typewell.csv'))
            _tvt_in = _hw['TVT_input'].to_numpy(dtype=float)
            _toe = ~_bnp.isfinite(_tvt_in)
            if 'TVT' not in _hw.columns or int(_toe.sum()) < 20:
                continue
            _true = _hw['TVT'].to_numpy(dtype=float)
            _kn = _hw[_hw['TVT_input'].notna()]
            if len(_kn) < 30:
                continue
            _pf_by_scale = run_pf_lik_ensemble_scales(
                _hw, _tw, scales=tuple(globals().get('SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0))),
                n_particles=_BO_PART, n_seeds=_BO_SEEDS)
            try:
                _beam = run_beam_ensemble(_hw, _tw)
            except Exception:
                _beam = _pf_by_scale.get('pf_mean', next(iter(_pf_by_scale.values())))
            _lk = float(_kn['TVT_input'].iloc[-1])
            _variant = selector_well_code(_hw)[1]
            _sel = _bnp.asarray(apply_selector_variant(_variant, _pf_by_scale, _beam, _lk), dtype=float)
            _ti = _bnp.flatnonzero(_toe)
            _ids = [f'{_wid}_{int(_i)}' for _i in _ti]
            _s1 = _bnp.asarray([_bo_sub1.get(_x, _bnp.nan) for _x in _ids], dtype=float)
            _fl = _bnp.asarray([_bo_fle.get(_x, _bnp.nan) for _x in _ids], dtype=float)
            _tc = _bnp.asarray([_bo_tcnm.get(_x, _bnp.nan) for _x in _ids], dtype=float)
            _yt = _true[_ti]
            _s2 = _sel[_ti]
            _mk = (_bnp.isfinite(_yt) & _bnp.isfinite(_s1) & _bnp.isfinite(_s2)
                   & _bnp.isfinite(_tc) & (_bnp.isfinite(_fl) if _bo_fle else True))
            if not _bo_fle:
                _fl = _s1                              # placeholder; FINAL views are skipped
            if int(_mk.sum()) < 20:
                continue
            _lr = _kn.iloc[-1]
            _bo_wells.append(dict(
                wid=_wid, true=_yt[_mk], s1=_s1[_mk], s2=_s2[_mk], fle=_fl[_mk], tcn=_tc[_mk],
                Z=_hw['Z'].to_numpy(dtype=float)[_ti][_mk],
                MD=_hw['MD'].to_numpy(dtype=float)[_ti][_mk],
                anchor=float(_lr['TVT_input']) + float(_lr['Z']),
                ps=float(_lr['MD']), end=float(_hw['MD'].iloc[-1]), nk=int(len(_kn)),
                hw=_hw, tw=_tw, beam=_beam, variant=_variant, lk=float(_lk), ti=_ti, mk=_mk))
            if (_bi + 1) % 10 == 0:
                print(f'[BLEND-OOF] ... {_bi + 1}/{len(_bo_wids)} wells, '
                      f'{_bo_time.time() - _bo_t0:.0f}s elapsed', flush=True)
        except Exception as _eb:
            print('[BLEND-OOF] well fail', _wid, str(_eb)[:80])
    print(f'[BLEND-OOF] wells usable={len(_bo_wells)} ({_bo_time.time() - _bo_t0:.0f}s)', flush=True)

    def _bo_proj(w, series):
        # the production projection cell, verbatim per well (deg=4, 0.25 raw + 0.75 fit)
        if len(series) < 5 or w['nk'] < 5:
            return series
        _s = (w['MD'] - w['ps']) / max(w['end'] - w['ps'], 1e-6)
        _fit = _robfit(_s, (series + w['Z']) - w['anchor'], 4)
        _pr = 0.25 * series + 0.75 * ((w['anchor'] + _fit) - w['Z'])
        return _pr if bool(_bnp.all(_bnp.isfinite(_pr))) else series

    def _bo_rmse(a, b):
        return float(_bnp.sqrt(_bnp.mean((_bnp.asarray(a, dtype=float) - _bnp.asarray(b, dtype=float)) ** 2)))

    def _bo_eval(pred_of):
        # per-well RMSE list + POOLED RMSE (same aggregation as cv_final / the LB metric)
        _errs = []
        _ss = 0.0
        _n = 0
        for _w in _bo_wells:
            _r = _bnp.asarray(pred_of(_w), dtype=float) - _w['true']
            _errs.append(float(_bnp.sqrt(_bnp.mean(_r ** 2))))
            _ss += float((_r ** 2).sum())
            _n += len(_r)
        return _bnp.asarray(_errs, dtype=float), float(_bnp.sqrt(_ss / max(_n, 1)))

    def _bo_final_pred(w, inner, sp45, wtcn, proj=True):
        _sp = inner * w['s2'] + (1.0 - inner) * w['s1']
        if proj:
            _sp = _bo_proj(w, _sp)
        return (1.0 - wtcn) * (sp45 * _sp + (1.0 - sp45) * w['fle']) + wtcn * w['tcn']

    def _bo_stats(e):
        _e = _bnp.sort(_bnp.asarray(e, dtype=float))
        _tm = float(_e[:-2].mean()) if len(_e) > 4 else float(_e.mean())
        return float(_e.mean()), float(_bnp.median(_e)), _tm

    def _bo_line(tag, e, pooled):
        _m, _md, _tm = _bo_stats(e)
        print(f'[BLEND-OOF] {tag:36s} pooled={pooled:7.4f} mean={_m:7.4f} '
              f'median={_md:7.4f} trim2={_tm:7.4f}', flush=True)

    if _bo_wells:
        for _key, _lab in (('s1', 'comp sub1 ridge-pp OOF'), ('s2', 'comp sub2 selector@prod'),
                           ('tcn', 'comp tcn OOF')):
            _e, _pl = _bo_eval(lambda w, k=_key: w[k])
            _bo_line(_lab, _e, _pl)
        _e, _pl = _bo_eval(lambda w: _bo_proj(w, 0.7 * w['s2'] + 0.3 * w['s1']))
        _bo_line('comp sp45(proj) inner=0.7', _e, _pl)
    if _bo_wells and _bo_fle:
        _e, _pl = _bo_eval(lambda w: w['fle'])
        _bo_line('comp fleongg OOF', _e, _pl)
        _ref, _refp = _bo_eval(lambda w: _bo_final_pred(w, 0.70, 0.55, 0.15))
        _bo_line('FINAL@default 0.70/0.55/0.15', _ref, _refp)
        _e, _pl = _bo_eval(lambda w: _bo_final_pred(w, 0.70, 0.55, 0.15, proj=False))
        _bo_line('FINAL@default no-projection', _e, _pl)
        _fold = _bnp.arange(len(_bo_wells)) % 2

        def _bo_sweep(tag, grid, fn, defv):
            for _v in grid:
                _e, _pl = _bo_eval(fn(_v))
                _m, _md, _tm = _bo_stats(_e)
                _mA = _bo_stats(_e[_fold == 0])[0] if int((_fold == 0).sum()) else float('nan')
                _mB = _bo_stats(_e[_fold == 1])[0] if int((_fold == 1).sum()) else float('nan')
                _win = int((_e < _ref - 1e-9).sum())
                _lose = int((_e > _ref + 1e-9).sum())
                _mark = '  <-default' if abs(_v - defv) < 1e-9 else ''
                print(f'[BLEND-OOF] {tag}={_v:.2f} pooled={_pl:7.4f} mean={_m:7.4f} '
                      f'median={_md:7.4f} trim2={_tm:7.4f} win/lose={_win}/{_lose} '
                      f'foldA={_mA:.4f} foldB={_mB:.4f}{_mark}', flush=True)

        _bo_sweep('w_inner', _BO_INNER_GRID, lambda v: (lambda w: _bo_final_pred(w, v, 0.55, 0.15)), 0.70)
        _bo_sweep('w_sp45 ', _BO_SP45_GRID, lambda v: (lambda w: _bo_final_pred(w, 0.70, v, 0.15)), 0.55)
        _bo_sweep('w_tcn  ', _BO_TCN_GRID, lambda v: (lambda w: _bo_final_pred(w, 0.70, 0.55, v)), 0.15)
        _ord = _bnp.argsort(-_ref)[:5]
        for _i in _ord:
            _w = _bo_wells[int(_i)]
            print(f'[BLEND-OOF] drift {_w["wid"]} FINAL={_ref[int(_i)]:.3f} '
                  f's1={_bo_rmse(_w["s1"], _w["true"]):.3f} s2={_bo_rmse(_w["s2"], _w["true"]):.3f} '
                  f'fle={_bo_rmse(_w["fle"], _w["true"]):.3f} tcn={_bo_rmse(_w["tcn"], _w["true"]):.3f}',
                  flush=True)
    elif _bo_wells:
        print('[BLEND-OOF] fleongg stash missing -> FINAL/sweep views skipped', flush=True)

    # --- [PF-SWEEP] recompute sub2 (selector-PF) per PF-motion-constant candidate ------------
    #     Hold s1/fle/tcn and the default weights (0.70/0.55/0.15); score FINAL vs held-out truth.
    #     beam/variant/lk are PF-independent -> cached in the well dict; only the PF is recomputed.
    #     CRN: run_pf_lik_ensemble_scales reseeds 0..n_seeds identically per candidate == paired delta.
    #     SCREEN ONLY: fle is frozen (mounted) -> a winner must be re-certified with fle unfrozen.
    if _bo_wells and _bo_fle:
        _PF_CANDS = ([('default_check', {}), ('MOM1.0', dict(MOM=1.0)), ('GS_LO5', dict(GS_LO=5.0))]
                     if SMOKE else
                     [('default_check',   {}),
                      ('MOM0.995',        dict(MOM=0.995)),
                      ('MOM0.999',        dict(MOM=0.999)),
                      ('MOM1.0',          dict(MOM=1.0)),
                      ('MOM1.0_VN0.004',  dict(MOM=1.0, VN=0.004)),
                      ('GS_LO5',          dict(GS_LO=5.0)),
                      ('GS_LO7',          dict(GS_LO=7.0))])
        _pfsc = tuple(globals().get('SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0)))
        _pf_fold = _bnp.arange(len(_bo_wells)) % 2
        _rarr = _bnp.asarray(_ref, dtype=float)             # baseline FINAL@default per-well RMSE
        _pf_drift = _rarr >= 9.0                            # drift wells = high baseline FINAL RMSE
        import time as _pf_time
        # cache baseline FINAL per-well residual-sums for the fast leave-one-out recompute
        _base_ss = _bnp.asarray([float(((_bo_final_pred(_w, 0.70, 0.55, 0.15) - _w['true']) ** 2).sum())
                                 for _w in _bo_wells], dtype=float)
        _base_n = _bnp.asarray([len(_w['true']) for _w in _bo_wells], dtype=float)
        print(f'[PF-SWEEP] {len(_PF_CANDS)} candidates x {len(_bo_wells)} wells @ {_BO_SEEDS}x{_BO_PART} '
              f'(baseline pooled={_refp:.4f})', flush=True)
        try:
            import resource as _pf_res
            print(f'[PF-SWEEP] maxRSS={_pf_res.getrusage(_pf_res.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB '
                  f'before sweep', flush=True)
        except Exception:
            pass
        for _cname, _cand in _PF_CANDS:
            _pf_t0 = _pf_time.time(); _nfb = 0
            _cfin = []
            for _w in _bo_wells:
                try:
                    _pf2 = run_pf_lik_ensemble_scales(_w['hw'], _w['tw'], scales=_pfsc,
                                                      n_particles=_BO_PART, n_seeds=_BO_SEEDS, **_cand)
                    _sel2 = _bnp.asarray(apply_selector_variant(_w['variant'], _pf2, _w['beam'], _w['lk']),
                                         dtype=float)
                    _wc = dict(_w); _wc['s2'] = _sel2[_w['ti']][_w['mk']]
                    _cfin.append(_bnp.asarray(_bo_final_pred(_wc, 0.70, 0.55, 0.15), dtype=float))
                except Exception as _pe:
                    _nfb += 1
                    _cfin.append(_bnp.asarray(_bo_final_pred(_w, 0.70, 0.55, 0.15), dtype=float))
            _perr = _bnp.empty(len(_bo_wells)); _cand_ss = _bnp.empty(len(_bo_wells)); _ss = 0.0; _ntot = 0
            for _wi, _w in enumerate(_bo_wells):
                _r = _cfin[_wi] - _w['true']
                _cand_ss[_wi] = float((_r ** 2).sum())
                _perr[_wi] = float(_bnp.sqrt(_bnp.mean(_r ** 2))); _ss += _cand_ss[_wi]; _ntot += len(_r)
            _cp = float(_bnp.sqrt(_ss / max(_ntot, 1)))
            _win = int((_perr < _rarr - 1e-9).sum()); _lose = int((_perr > _rarr + 1e-9).sum())
            _cm, _cmd, _ = _bo_stats(_perr); _rm, _rmd, _ = _bo_stats(_rarr)
            _mA = float(_bnp.mean(_perr[_pf_fold == 0])); _mB = float(_bnp.mean(_perr[_pf_fold == 1]))
            _rA = float(_bnp.mean(_rarr[_pf_fold == 0])); _rB = float(_bnp.mean(_rarr[_pf_fold == 1]))
            _full_d = _refp - _cp                            # >0 == candidate better (lower pooled)
            # leave-one-well-out sign stability: drop each well, require the pooled-delta sign to hold
            _tot_ss_c = _cand_ss.sum(); _tot_ss_b = _base_ss.sum(); _tot_n = _base_n.sum()
            _loo_ok = True
            for _j in range(len(_bo_wells)):
                _nn = _tot_n - _base_n[_j]
                if _nn <= 0:
                    continue
                _dj = (_bnp.sqrt((_tot_ss_b - _base_ss[_j]) / _nn)
                       - _bnp.sqrt((_tot_ss_c - _cand_ss[_j]) / _nn))
                if (_full_d > 0) != (_dj > 0):
                    _loo_ok = False; break
            _dc = float(_bnp.sqrt((_perr[_pf_drift] ** 2).mean())) if _pf_drift.any() else float('nan')
            _rdc = float(_bnp.sqrt((_rarr[_pf_drift] ** 2).mean())) if _pf_drift.any() else float('nan')
            _cc = float(_bnp.sqrt((_perr[~_pf_drift] ** 2).mean())) if (~_pf_drift).any() else float('nan')
            _rcc = float(_bnp.sqrt((_rarr[~_pf_drift] ** 2).mean())) if (~_pf_drift).any() else float('nan')
            print(f'[PF-SWEEP] {_cname:16s} pooled={_cp:7.4f}(d={_full_d:+.4f}) '
                  f'median={_cmd:7.4f}(def {_rmd:.4f}) win/lose={_win}/{_lose} '
                  f'foldA={_mA:.3f}/{_rA:.3f} foldB={_mB:.3f}/{_rB:.3f} '
                  f'LOOsign={"OK" if _loo_ok else "FLIP"} '
                  f'drift={_dc:.3f}/{_rdc:.3f} clean={_cc:.3f}/{_rcc:.3f} '
                  f'fb={_nfb} ({_pf_time.time()-_pf_t0:.0f}s)', flush=True)

    # --- [BLEND-OOF-FLE] fleongg w_sub1 sweep on ALL train wells (OOF-only, no physics) ---
    if _ST is not None:
        from scipy.signal import savgol_filter as _bo_sg
        _fid = _bpd.DataFrame({'well': _bpd.Series(_ST['id']).str[:8], 'true': _ST['true'],
                               'last': _ST['last'], 'sub1': _ST['sub1'], 'lp': _ST['lp']})
        for _wv in (0.40, 0.50, 0.55, 0.60, 0.68, 0.75, 0.85):
            _pr = _fid['last'].values + _wv * _fid['sub1'].values + (1.0 - _wv) * _fid['lp'].values
            _out = _pr.copy()
            for _, _ix in _fid.groupby('well', sort=False).groups.items():
                _pos = _fid.index.get_indexer(_ix)
                _v = _pr[_pos]
                _wl = min(PP.sg_win, len(_v))
                if _wl % 2 == 0:
                    _wl -= 1
                if _wl >= PP.sg_poly + 2:
                    _out[_pos] = _bo_sg(_v, _wl, PP.sg_poly)
            _pw = _bpd.DataFrame({'well': _fid['well'], 'e2': (_out - _fid['true'].values) ** 2})
            _pwr = _pw.groupby('well')['e2'].mean().pow(0.5)
            _mark = '  <-default' if abs(_wv - float(PP.w_sub1)) < 1e-9 else ''
            print(f'[BLEND-OOF-FLE] w_sub1={_wv:.2f} pooled={float(_bnp.sqrt(_pw["e2"].mean())):.4f} '
                  f'perwell_mean={float(_pwr.mean()):.4f} perwell_median={float(_pwr.median()):.4f} '
                  f'n_wells={len(_pwr)}{_mark}', flush=True)
except Exception as _ebo:
    print('[BLEND-OOF] SKIPPED:', str(_ebo)[:200])
